# Week2_ex1_vertex_check

Queries the polygon cross-section vertices of the `line` object to check whether the field_line direction (+y at the wire's mid-plane, angle = 90deg) lines up with a flat face or a vertex of the 12-sided circle approximation -- testing Hypothesis 1 for the near-surface sawtooth anomaly in `Week2_ex1_surface_discretization_analysis.ipynb`.


In [ ]:
import ansys.aedt.core
import numpy as np
import os


In [ ]:
## connect to the already-solved project ##

ansys.aedt.core.settings.enable_screen_logs = False

DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

proj_name = "Week2_ex1"
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
project_path = os.path.join(base_dir, proj_name, f"{proj_name}.aedt")

M3D = ansys.aedt.core.Maxwell3d(project=project_path, design="Week2_ex1", remove_lock=True)


In [ ]:
## pull the vertex coordinates of the line object -- these sit at the two end caps of the swept circle ##

line_obj = M3D.modeler["line"]
verts = line_obj.vertices

# keep only the vertices at one end cap (x close to -line_length/2 = -50mm), the other cap is a mirror
end_cap_verts = [v.position for v in verts if abs(v.position[0] - (-50.0)) < 0.001]

end_cap_verts


In [ ]:
## compute the angular position of each vertex around the wire axis (y-z plane, angle from +y) ##

angles_deg = [np.degrees(np.arctan2(v[2], v[1])) % 360 for v in end_cap_verts]
sorted(angles_deg)


In [ ]:
## field_line runs along +y, i.e. angle = 90deg -- check how close that is to a vertex vs a flat face midpoint ##
## (0deg gap = field_line hits a vertex exactly, 15deg gap = field_line hits a flat face center exactly, since 360/12 = 30deg spacing) ##

target_angle = 90.0
closest_gap = min(abs(((a - target_angle + 180) % 360) - 180) for a in angles_deg)
closest_gap


In [ ]:
## release without saving -- pure geometry query, no design variables were touched ##

M3D.release_desktop(close_projects=False, close_desktop=False)
